In [2]:
from pyspark import SparkContext

In [3]:
from pyspark.sql import SparkSession #w jednym notatniku/systemie mozna miec tylko jedna sesje sparka

In [4]:
spark = SparkSession.builder.getOrCreate()

In [5]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
#to samo w terminalu poprzez wpisanie pyspark

In [6]:
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [7]:
sc = spark.sparkContext #skrot

In [8]:
df = spark.read.json("transactions_10k.jsonl") 
#ten plik jest w tym samym folderze co sparksession i dlatego dajemy tylko nazwę #pliki json, tekstowe

print(f"Liczba rekordów: {df.count()}")
df.printSchema() #schemat czyli okreslenie jakiego typu dana kolumna jest

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [9]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [10]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)


In [11]:
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [14]:
#zadanie 2.2 - w domu
from pyspark.sql.functions import sum as _sum, min as _min, max as _max
category_summary = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN"),
    )
    .orderBy("category")
)
category_summary.show()

+-----------+----------+-------+-------+
|   category|  suma_PLN|min_PLN|max_PLN|
+-----------+----------+-------+-------+
|elektronika|1520770.69|    9.0| 9999.0|
|    książki| 851382.08|    5.0|9107.25|
|     odzież| 849877.55|    5.0|9696.63|
|    żywność| 789514.43|    5.0|6916.92|
+-----------+----------+-------+-------+



In [20]:
from pyspark.sql.functions import window


In [17]:
hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne #window(kolumna, zakres czasowy)
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [18]:
hourly.printSchema()
#struct to jest cos co w srodku zawiera dodatkwoe elementy i te elementy maja swoje nazwy/rodzaje

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- liczba_tx: long (nullable = false)
 |-- suma_PLN: double (nullable = true)



In [19]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [24]:
#zadanie 3.2 - samodzielnie
half_hourly_store = (
    df.groupBy(window("timestamp", "30 minutes"), "store")    # okno 30min #window(kolumna, zakres czasowy)
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window", "store")
)
half_hourly_store.show(truncate=False)

+------------------------------------------+--------+---------+---------+
|window                                    |store   |liczba_tx|suma_PLN |
+------------------------------------------+--------+---------+---------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Gdańsk  |252      |93391.22 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Kraków  |289      |117786.42|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Warszawa|275      |88441.58 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Wrocław |296      |111540.59|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Gdańsk  |514      |209187.85|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Kraków  |532      |223541.41|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Warszawa|490      |182435.06|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Wrocław |502      |215587.17|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Gdańsk  |619      |253364.95|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Kraków  |590      |224358.03|
|{2026-04-12 09:00:00, 2026-04-12 09:3

In [25]:
#zadanie 3.3 - samodzielnie
from pyspark.sql.functions import col, window, sum, round, desc

# TWÓJ KOD
krakow_top_hour = (
    df.filter(col("store") == "Kraków")                 # 1. Filtruj najpierw po sklepie
      .groupBy(window("timestamp", "1 hour"))           # 2. Zrób okno godzinne
      .agg(round(sum("amount"), 2).alias("suma_PLN"))   # 3. Policz sumę przychodów
      .orderBy(desc("suma_PLN"))                        # 4. Posortuj malejąco po sumie
)

krakow_top_hour.show(truncate=False)

+------------------------------------------+---------+
|window                                    |suma_PLN |
+------------------------------------------+---------+
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|483309.86|
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|341327.83|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|201259.26|
+------------------------------------------+---------+



In [16]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [17]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [ ]:
#Część 5: Pytania kontrolne
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: Zwykłe groupBy("store") agreguje dane "globalnie". Zwróci nam tylko jeden wiersz dla każdego sklepu 
#      (np. absolutnie całą sumę przychodów dla Krakowa od początku działania programu).
#    - groupBy(window(...), "store") dzieli dane dodatkowo na przedziały czasowe. Zwróci nam wiele wierszy 
#      dla jednego sklepu (np. osobną sumę dla Krakowa z godziny 12:00-13:00, osobną z 13:00-14:00 itd.).

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: Zawierają ją dokładnie 2 okna.
#    Wynika to z prostej matematyki okien przesuwnych: Szerokość okna (60 min) / Krok (30 min) = 2.
#    Rozpisując na osi czasu dla transakcji z godziny 09:30, wpadnie ona do:
#    1. Okna: 09:00 - 10:00
#    2. Okna: 09:30 - 10:30
#    (Uwaga techniczna: okno 08:30-09:30 jej NIE zawiera, ponieważ w Sparku okna domyślnie 
#     nie wliczają swojej górnej granicy, tzn. działają na zasadzie [od, do) ).

In [ ]:
#Praca domowa

In [34]:
#1. Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.
from pyspark.sql.functions import col, window, avg, round

# TWÓJ KOD
gdansk_worst_hour = (
    df.filter(col("store") == "Gdańsk")                 # 1. Filtruj najpierw po sklepie
      .groupBy(window("timestamp", "1 hour"))           # 2. Zrób okno godzinne
      .agg(round(avg("amount"), 2).alias("srednia_PLN"))   # 3. Policz sumę przychodów
      .orderBy("srednia_PLN")                        # 4. Posortuj malejąco po sumie
)

gdansk_worst_hour.show(truncate=False)

#odp: 08:00 - 09:00

+------------------------------------------+-----------+
|window                                    |srednia_PLN|
+------------------------------------------+-----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|395.01     |
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|412.92     |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|415.91     |
+------------------------------------------+-----------+



In [33]:
#2.Policz ile transakcji per kategoria było w oknie 09:00–09:30.
from pyspark.sql.functions import sum as _sum, min as _min, max as _max
category_summary = (
    df.groupBy("category", window("timestamp", "30 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
    )
    .filter(col("window.start") == "2026-04-12 09:00:00")
    .orderBy("window","category")
)
category_summary.show()

#odp: elektronika - 611, książki - 622, odzież - 605, żywność - 567

+-----------+--------------------+---------+
|   category|              window|liczba_tx|
+-----------+--------------------+---------+
|elektronika|{2026-04-12 09:00...|      611|
|    książki|{2026-04-12 09:00...|      622|
|     odzież|{2026-04-12 09:00...|      605|
|    żywność|{2026-04-12 09:00...|      567|
+-----------+--------------------+---------+



In [39]:
#3.Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).
from pyspark.sql.functions import window, desc

cwiercgodz = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
    )
    .orderBy(desc("liczba_tx"))
)
cwiercgodz.show(truncate=False)

#odp: 09:15 - 09:30

+------------------------------------------+---------+
|window                                    |liczba_tx|
+------------------------------------------+---------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234     |
|{2026-04-12 09:00:00, 2026-04-12 09:15:00}|1171     |
|{2026-04-12 09:30:00, 2026-04-12 09:45:00}|1156     |
|{2026-04-12 08:45:00, 2026-04-12 09:00:00}|1139     |
|{2026-04-12 09:45:00, 2026-04-12 10:00:00}|1100     |
|{2026-04-12 08:30:00, 2026-04-12 08:45:00}|899      |
|{2026-04-12 10:00:00, 2026-04-12 10:15:00}|858      |
|{2026-04-12 08:15:00, 2026-04-12 08:30:00}|644      |
|{2026-04-12 10:15:00, 2026-04-12 10:30:00}|582      |
|{2026-04-12 08:00:00, 2026-04-12 08:15:00}|468      |
|{2026-04-12 10:30:00, 2026-04-12 10:45:00}|443      |
|{2026-04-12 10:45:00, 2026-04-12 11:00:00}|306      |
+------------------------------------------+---------+

